In [ ]:
import numpy as np
import SimpleITK as sitk
import pandas as pd
import os
from pathlib import Path
from skimage import morphology
from skimage.measure import regionprops
from scipy.ndimage import label, generate_binary_structure
import cv2
from totalsegmentator.python_api import totalsegmentator
from surface_distance import (
    compute_surface_distances,
    compute_average_surface_distance,
    compute_surface_dice_at_tolerance,
    compute_robust_hausdorff,
)
from matplotlib import pyplot as plt
%matplotlib inline

validation_root_path = os.path.join(os.getcwd(), r'Images\Validation Dataset')
validation_dataset_paths = {
    'kits19': os.path.join(validation_root_path, 'KiTS19'),
    'msd_t10': os.path.join(validation_root_path, 'MSD_T10'),
}

working_spacing_mm = 1.5
totalseg_device = 'gpu'
overwrite = False


In [ ]:
def _binary_threshold_np(img, low, high):
    arr = np.asarray(img)
    out = np.zeros(arr.shape, dtype=np.uint8)
    m = np.isfinite(arr)
    out[m] = ((arr[m] >= low) & (arr[m] <= high)).astype(np.uint8)
    return out


def _remove_small_objects_np(mask, min_size):
    lab = morphology.label(mask)
    cleaned = morphology.remove_small_objects(lab, min_size)
    return (cleaned > 0).astype(np.uint8)


def _floodfill_np(mask):
    arr = np.asarray(mask, dtype=np.uint8)
    h, w = arr.shape
    ff = arr.copy()
    m = np.zeros((h+2, w+2), np.uint8)
    cv2.floodFill(ff, m, (0,0), 255)
    inv = (cv2.bitwise_not(ff) > 0).astype(np.uint8)
    return inv


def _dilate_np(mask, kernel, it):
    m = np.asarray(mask, dtype=np.uint8)
    k = np.asarray(kernel, dtype=np.uint8)
    i = int(np.asarray(it, dtype=np.uint8))
    return cv2.dilate(m, k, iterations=i)


def _erode_np(mask, kernel, it):
    m = np.asarray(mask, dtype=np.uint8)
    k = np.asarray(kernel, dtype=np.uint8)
    i = int(np.asarray(it, dtype=np.uint8))
    return cv2.erode(m, k, iterations=i)


def clean_mask_hip(ct_arr, ct_cropped, hip_mask, dilate_kernel=None, dilate_iters=1, erode_iters=1):
    if dilate_kernel is None:
        dilate_kernel = np.array([[0,1,0],[1,1,1],[0,1,0]], dtype=np.uint8)

    zdim = hip_mask.shape[0]
    refined_mask = (hip_mask > 0).astype(np.uint8)
    
    # Metrics holder for HU-based expansion
    added_voxels_records = np.zeros(zdim, dtype=np.int32)

    # Find inferior bound of acetabulum corresponding to 0.6 cm below the upper border of obturator foramen
    # The longest run of slices >= 2 objects correspond to obturator foramen
    max_len = 0
    current_len = 0
    end_idx = -1
    for z in range(zdim):
        cleaned_slice = _remove_small_objects_np(hip_mask[z], min_size=100)
        _, count = label(cleaned_slice)
        if count >= 2:
            current_len += 1
            if current_len > max_len:
                max_len = current_len
                end_idx = z
        else:
            current_len = 0

    if max_len > 0:
        start_idx = end_idx - max_len + 1
        print(f"Longest run of slices with ≥2 objects: {max_len} slices (z={start_idx} to z={end_idx})")
        print(f"Most superior slice of obturator foramen: z={end_idx}")
    
        inferior_bound = max(end_idx - 3, 0)
        print(f"Inferior bound: z={inferior_bound}")
    else:
        print("Obturator foramen not detected -- falling back to most inferior non-empty slice.")
        
        inferior_bound = None
        for z in range(zdim):
            cleaned_slice = _remove_small_objects_np(hip_mask[z], min_size=100)
            if np.any(cleaned_slice):
                inferior_bound = z
                break

        if inferior_bound is None:
            print("Hip bone mask is empty -- returning original mask.")
            return refined_mask, None, None, added_voxels_records

        print(f"Inferior bound (fallback): z={inferior_bound}")

    # Find superior bound corresponding to 0.9 cm above the acetabular roof
    # The first slice with > 0.9 solidity corresponds to acetabular roof
    superior_bound = None
    for z in range(end_idx + 1, zdim):
        labeled, _ = label(hip_mask[z])
        props = regionprops(labeled)
        if props:
            largest_region = max(props, key=lambda r: r.area)
            solidity = largest_region.solidity
        else:
            solidity = 0.0

        if solidity > 0.9:
            superior_bound = z + 6
            if superior_bound >= zdim:
                superior_bound = zdim - 1
            print(f"Acetabular roof at slice {z}, setting superior bound = {superior_bound}")
            break

    if superior_bound is None:
        superior_bound = inferior_bound + 34
        if superior_bound >= zdim:
            superior_bound = zdim - 1
        print("No slice with solidity > 0.9 found. Forcing range to be 34.")

    # Sanity check: valid cleaning range should be within 4.2-6.0 cm
    cleaning_range = superior_bound - inferior_bound
    if cleaning_range < 28 or cleaning_range > 40:
        print(f"Warning: suspicious acetabular roof identification with {cleaning_range} slices of acetabulum. Forcing range to be 34.")
        superior_bound = inferior_bound + 34
        if superior_bound >= zdim:
            superior_bound = zdim - 1

    print(f"Cleaning range: z={inferior_bound} to z={superior_bound}")
    
    # Clean selected slices
    for i in range(inferior_bound, min(superior_bound + 1, zdim)):
        slice_crop = ct_cropped[i]
        slice_ct = ct_arr[i]

        # Apply binary thresholding
        mask_bi = _binary_threshold_np(slice_crop, low=150, high=np.inf)

        # HU-based expansion
        max_added_voxels = 40  # sanity cap per slice
        high_hu = slice_ct > 300
        labeled_high, num_high = label(high_hu.astype(np.uint8))
    
        if num_high > 0:
            touching_labels = np.unique(labeled_high[mask_bi.astype(bool)])
            touching_labels = touching_labels[touching_labels != 0]
    
            if touching_labels.size > 0:
                connected_high = np.isin(labeled_high, touching_labels)
                expanded_mask = mask_bi.astype(bool) | connected_high
                added_voxels = (np.count_nonzero(expanded_mask) - np.count_nonzero(mask_bi))
                
                # Record metric
                added_voxels_records[i] = int(added_voxels)
                
                if added_voxels <= max_added_voxels:
                    mask_exp = expanded_mask.astype(np.uint8)
                else:
                    print(
                        f"Slice {i}: HU-based expansion added {added_voxels} voxels "
                        f"(> {max_added_voxels}); reverting to mask after binary thresholding."
                    )
                    mask_exp = mask_bi
            else:
                mask_exp = mask_bi
        else:
            mask_exp = mask_bi
        mask_dil = _dilate_np(mask_exp, dilate_kernel, dilate_iters)
        mask_erod = _erode_np(mask_dil, dilate_kernel, erode_iters)
        mask_fill = _floodfill_np(mask_erod)
        refined_mask[i] = mask_fill
        
    # 3D connectivity: keep only the largest connected component
    structure = generate_binary_structure(3, 1)
    labeled_3d, num = label(refined_mask.astype(bool), structure=structure)
    print(f"3D connected components found: {num}")
    if num == 0:
        print("Warning: refined hip mask is empty after 3D labeling.")
        return refined_mask, inferior_bound, superior_bound, added_voxels_records
    
    counts = np.bincount(labeled_3d.ravel())
    counts[0] = 0
    
    largest_label = counts.argmax()
    refined_mask = (labeled_3d == largest_label).astype(np.uint8)
    return refined_mask, inferior_bound, superior_bound, added_voxels_records

In [ ]:
def compute_voxel_count_diff(
    right_hip_mask_cleaned,
    right_hip_mask_cleaned_fallback,
    right_hip_pred_arr_rev_bi,
    left_hip_mask_cleaned,
    left_hip_mask_cleaned_fallback,
    left_hip_pred_arr_rev_bi,
    right_added_voxels=None,
    left_added_voxels=None,
    thresh_fallback_replace=60,
    thresh_orig_lower=-10,
    thresh_orig_upper=200,
):

    assert right_hip_mask_cleaned.shape == right_hip_mask_cleaned_fallback.shape == right_hip_pred_arr_rev_bi.shape
    assert left_hip_mask_cleaned.shape == left_hip_mask_cleaned_fallback.shape == left_hip_pred_arr_rev_bi.shape

    zdim = right_hip_mask_cleaned.shape[0]
    assert left_hip_mask_cleaned.shape[0] == zdim

    if right_added_voxels is None:
        right_added_voxels = np.zeros(zdim, dtype=np.int32)
    if left_added_voxels is None:
        left_added_voxels = np.zeros(zdim, dtype=np.int32)
    
    rows = []

    # Voxel count difference-based overrule: cleaned vs fallback
    right_cleaned_count = np.count_nonzero(right_hip_mask_cleaned, axis=(1, 2)).astype(np.int32)
    right_fallback_count = np.count_nonzero(right_hip_mask_cleaned_fallback, axis=(1, 2)).astype(np.int32)
    right_diff1 = right_fallback_count - right_cleaned_count
    right_replace_fallback = right_diff1 > thresh_fallback_replace
    left_cleaned_count = np.count_nonzero(left_hip_mask_cleaned, axis=(1, 2)).astype(np.int32)
    left_fallback_count = np.count_nonzero(left_hip_mask_cleaned_fallback, axis=(1, 2)).astype(np.int32)
    left_diff1 = left_fallback_count - left_cleaned_count
    left_replace_fallback = left_diff1 > thresh_fallback_replace
    right_hip_mask_processed = right_hip_mask_cleaned.copy()
    left_hip_mask_processed = left_hip_mask_cleaned.copy()
    right_hip_mask_processed[right_replace_fallback] = right_hip_mask_cleaned_fallback[right_replace_fallback]
    left_hip_mask_processed[left_replace_fallback] = left_hip_mask_cleaned_fallback[left_replace_fallback]

    # Voxel count difference-based overrule: processed vs original
    right_processed_count = np.count_nonzero(right_hip_mask_processed, axis=(1, 2)).astype(np.int32)
    right_original_count = np.count_nonzero(right_hip_pred_arr_rev_bi, axis=(1, 2)).astype(np.int32)
    right_diff2 = right_original_count - right_processed_count
    right_replace_original = (right_diff2 < thresh_orig_lower) | (right_diff2 > thresh_orig_upper)
    left_processed_count = np.count_nonzero(left_hip_mask_processed, axis=(1, 2)).astype(np.int32)
    left_original_count = np.count_nonzero(left_hip_pred_arr_rev_bi, axis=(1, 2)).astype(np.int32)
    left_diff2 = left_original_count - left_processed_count
    left_replace_original = (left_diff2 < thresh_orig_lower) | (left_diff2 > thresh_orig_upper)

    # Construct dataframe
    for z in range(zdim):
        rows.append({
            "side": "right",
            "slice_z": z,

            "count_cleaned": int(right_cleaned_count[z]),
            "count_fallback": int(right_fallback_count[z]),
            "diff_fallback_minus_cleaned": int(right_diff1[z]),
            "replaced_by_fallback": bool(right_replace_fallback[z]),

            "count_processed": int(right_processed_count[z]),
            "count_original": int(right_original_count[z]),
            "diff_original_minus_processed": int(right_diff2[z]),
            "replaced_by_original": bool(right_replace_original[z]),
            "added_voxels": int(right_added_voxels[z]),
        })

        rows.append({
            "side": "left",
            "slice_z": z,

            "count_cleaned": int(left_cleaned_count[z]),
            "count_fallback": int(left_fallback_count[z]),
            "diff_fallback_minus_cleaned": int(left_diff1[z]),
            "replaced_by_fallback": bool(left_replace_fallback[z]),

            "count_processed": int(left_processed_count[z]),
            "count_original": int(left_original_count[z]),
            "diff_original_minus_processed": int(left_diff2[z]),
            "replaced_by_original": bool(left_replace_original[z]),
            "added_voxels": int(left_added_voxels[z]),
        })

    df = pd.DataFrame(rows)
    return df

In [ ]:
def find_validation_inputs(input_path):
    case_name = os.path.basename(input_path)
    if case_name.startswith('case_'):
        image_path = os.path.join(input_path, 'imaging.nii.gz')
        ground_truth_path = os.path.join(
            input_path, f'dataset4_{case_name}_mask_4label.nii.gz'
        )
    else:
        image_path = os.path.join(input_path, f'{case_name}.nii.gz')
        ground_truth_path = os.path.join(
            input_path, f'dataset3_{case_name}_mask_4label.nii.gz'
        )
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f'expected validation image not found: {image_path}')
    if not os.path.isfile(ground_truth_path):
        raise FileNotFoundError(
            f'expected validation ground truth not found: {ground_truth_path}'
        )
    return image_path, ground_truth_path


def resample_to_isotropic(image, target_spacing=1.5, interpolator=sitk.sitkLinear):
    original_spacing = image.GetSpacing()
    original_size = image.GetSize()
    new_spacing = (target_spacing, target_spacing, target_spacing)
    new_size = [
        int(round(original_size[i] * (original_spacing[i] / new_spacing[i])))
        for i in range(3)
    ]
    resample = sitk.ResampleImageFilter()
    resample.SetOutputSpacing(new_spacing)
    resample.SetSize(new_size)
    resample.SetOutputDirection(image.GetDirection())
    resample.SetOutputOrigin(image.GetOrigin())
    resample.SetInterpolator(interpolator)
    return resample.Execute(image)


def resample_to_reference(
    image, reference_image, interpolator=sitk.sitkNearestNeighbor
):
    resample = sitk.ResampleImageFilter()
    resample.SetReferenceImage(reference_image)
    resample.SetInterpolator(interpolator)
    return resample.Execute(image)


def run_cleaning_workflow(input_path):
    image_path, ground_truth_path = find_validation_inputs(input_path)
    ct = sitk.ReadImage(image_path, sitk.sitkFloat32)
    ground_truth = sitk.ReadImage(ground_truth_path)

    totalsegmentator(
        image_path, input_path, task='total', roi_subset=['hip_right'],
        fast=False, ml=False, device=totalseg_device, skip_saving=False
    )
    totalsegmentator(
        image_path, input_path, task='total', roi_subset=['hip_left'],
        fast=False, ml=False, device=totalseg_device, skip_saving=False
    )
    right_mask = sitk.Cast(
        sitk.ReadImage(os.path.join(input_path, 'hip_right.nii.gz')) > 0,
        sitk.sitkUInt8,
    )
    left_mask = sitk.Cast(
        sitk.ReadImage(os.path.join(input_path, 'hip_left.nii.gz')) > 0,
        sitk.sitkUInt8,
    )

    ct_lps = sitk.DICOMOrient(ct, 'LPS')
    working_ct = resample_to_isotropic(
        ct_lps, target_spacing=working_spacing_mm,
        interpolator=sitk.sitkLinear
    )
    right_working = resample_to_reference(
        right_mask, working_ct, interpolator=sitk.sitkNearestNeighbor
    )
    left_working = resample_to_reference(
        left_mask, working_ct, interpolator=sitk.sitkNearestNeighbor
    )
    ct_arr = sitk.GetArrayFromImage(working_ct)
    right_hip_pred_arr_rev_bi = (
        sitk.GetArrayFromImage(right_working) > 0
    ).astype(np.uint8)
    left_hip_pred_arr_rev_bi = (
        sitk.GetArrayFromImage(left_working) > 0
    ).astype(np.uint8)
    right_hip_cropped = np.where(
        right_hip_pred_arr_rev_bi > 0, ct_arr, np.nan
    )
    left_hip_cropped = np.where(
        left_hip_pred_arr_rev_bi > 0, ct_arr, np.nan
    )

    print('\n=== right hip | default morphological closing ===')
    right_hip_mask_cleaned, right_inf, right_sup, right_added_voxels = clean_mask_hip(
        ct_arr, right_hip_cropped, right_hip_pred_arr_rev_bi,
        dilate_kernel=None, dilate_iters=1, erode_iters=1
    )
    print('\n=== left hip | default morphological closing ===')
    left_hip_mask_cleaned, left_inf, left_sup, left_added_voxels = clean_mask_hip(
        ct_arr, left_hip_cropped, left_hip_pred_arr_rev_bi,
        dilate_kernel=None, dilate_iters=1, erode_iters=1
    )
    print('\n=== right hip | fallback morphological closing ===')
    right_hip_mask_cleaned_fallback, _, _, _ = clean_mask_hip(
        ct_arr, right_hip_cropped, right_hip_pred_arr_rev_bi,
        dilate_kernel=None, dilate_iters=3, erode_iters=3
    )
    print('\n=== left hip | fallback morphological closing ===')
    left_hip_mask_cleaned_fallback, _, _, _ = clean_mask_hip(
        ct_arr, left_hip_cropped, left_hip_pred_arr_rev_bi,
        dilate_kernel=None, dilate_iters=3, erode_iters=3
    )

    right_hip_voxel_count = np.count_nonzero(
        right_hip_mask_cleaned, axis=(1, 2)
    ).astype(np.int32)
    right_hip_voxel_count_fallback = np.count_nonzero(
        right_hip_mask_cleaned_fallback, axis=(1, 2)
    ).astype(np.int32)
    left_hip_voxel_count = np.count_nonzero(
        left_hip_mask_cleaned, axis=(1, 2)
    ).astype(np.int32)
    left_hip_voxel_count_fallback = np.count_nonzero(
        left_hip_mask_cleaned_fallback, axis=(1, 2)
    ).astype(np.int32)
    right_slices_to_replace = np.where(
        right_hip_voxel_count_fallback - right_hip_voxel_count > 60
    )[0]
    left_slices_to_replace = np.where(
        left_hip_voxel_count_fallback - left_hip_voxel_count > 60
    )[0]
    right_hip_mask_processed = right_hip_mask_cleaned.copy()
    left_hip_mask_processed = left_hip_mask_cleaned.copy()
    right_hip_mask_processed[right_slices_to_replace] = (
        right_hip_mask_cleaned_fallback[right_slices_to_replace]
    )
    left_hip_mask_processed[left_slices_to_replace] = (
        left_hip_mask_cleaned_fallback[left_slices_to_replace]
    )

    right_processed_count = np.count_nonzero(
        right_hip_mask_processed, axis=(1, 2)
    ).astype(np.int32)
    left_processed_count = np.count_nonzero(
        left_hip_mask_processed, axis=(1, 2)
    ).astype(np.int32)
    right_original_count = np.count_nonzero(
        right_hip_pred_arr_rev_bi, axis=(1, 2)
    ).astype(np.int32)
    left_original_count = np.count_nonzero(
        left_hip_pred_arr_rev_bi, axis=(1, 2)
    ).astype(np.int32)
    right_diff = right_original_count - right_processed_count
    left_diff = left_original_count - left_processed_count
    right_slices_to_replace_final = np.where(
        (right_diff < -10) | (right_diff > 200)
    )[0]
    left_slices_to_replace_final = np.where(
        (left_diff < -10) | (left_diff > 200)
    )[0]
    right_hip_mask_cleaned_final = right_hip_mask_processed.copy()
    left_hip_mask_cleaned_final = left_hip_mask_processed.copy()
    right_hip_mask_cleaned_final[right_slices_to_replace_final] = (
        right_hip_pred_arr_rev_bi[right_slices_to_replace_final]
    )
    left_hip_mask_cleaned_final[left_slices_to_replace_final] = (
        left_hip_pred_arr_rev_bi[left_slices_to_replace_final]
    )

    # Resample the final cleaned masks back to the original grid
    right_working_cleaned = sitk.GetImageFromArray(
        right_hip_mask_cleaned_final.astype(np.uint8)
    )
    right_working_cleaned.CopyInformation(working_ct)
    left_working_cleaned = sitk.GetImageFromArray(
        left_hip_mask_cleaned_final.astype(np.uint8)
    )
    left_working_cleaned.CopyInformation(working_ct)
    right_mask_cleaned = resample_to_reference(
        right_working_cleaned, ct,
        interpolator=sitk.sitkNearestNeighbor
    )
    left_mask_cleaned = resample_to_reference(
        left_working_cleaned, ct,
        interpolator=sitk.sitkNearestNeighbor
    )
    sitk.WriteImage(
        right_mask_cleaned,
        os.path.join(input_path, 'hip_right_cleaned.nii.gz'),
    )
    sitk.WriteImage(
        left_mask_cleaned,
        os.path.join(input_path, 'hip_left_cleaned.nii.gz'),
    )

    outputs = {}
    for side, raw_image, cleaned_image, inferior_bound, superior_bound in (
        ('right', right_mask, right_mask_cleaned, right_inf, right_sup),
        ('left', left_mask, left_mask_cleaned, left_inf, left_sup),
    ):
        working_window_arr = np.zeros_like(ct_arr, dtype=np.uint8)
        working_window_arr[inferior_bound:superior_bound + 1] = 1
        working_window = sitk.GetImageFromArray(working_window_arr)
        working_window.CopyInformation(working_ct)
        native_window = resample_to_reference(
            working_window, ct,
            interpolator=sitk.sitkNearestNeighbor
        )
        outputs[side] = {
            'raw': raw_image,
            'cleaned': cleaned_image,
            'window': native_window,
        }

    return (
        ct,
        ground_truth,
        outputs,
        right_hip_mask_cleaned,
        right_hip_mask_cleaned_fallback,
        right_hip_pred_arr_rev_bi,
        left_hip_mask_cleaned,
        left_hip_mask_cleaned_fallback,
        left_hip_pred_arr_rev_bi,
        right_added_voxels,
        left_added_voxels,
        right_inf,
        right_sup,
        left_inf,
        left_sup,
    )


In [ ]:
def dice_formula(a, b, empty_score=1.0):
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    denominator = np.count_nonzero(a) + np.count_nonzero(b)
    if denominator == 0:
        return float(empty_score)
    return float(2 * np.count_nonzero(a & b) / denominator)


def compute_side_metrics(
    ground_truth,
    mask_original,
    mask_cleaned,
    acetabular_window,
    spacing,
):
    ground_truth = ground_truth & acetabular_window
    mask_original = mask_original & acetabular_window
    mask_cleaned = mask_cleaned & acetabular_window
    surface_original = compute_surface_distances(
        ground_truth, mask_original, spacing_mm=spacing
    )
    surface_cleaned = compute_surface_distances(
        ground_truth, mask_cleaned, spacing_mm=spacing
    )
    return {
        'dice_orig': dice_formula(ground_truth, mask_original),
        'dice_clean': dice_formula(ground_truth, mask_cleaned),
        'sd_orig': float(compute_surface_dice_at_tolerance(surface_original, 3.0)),
        'sd_clean': float(compute_surface_dice_at_tolerance(surface_cleaned, 3.0)),
        'asd_orig': float(np.mean(compute_average_surface_distance(surface_original))),
        'asd_clean': float(np.mean(compute_average_surface_distance(surface_cleaned))),
        'hd95_orig': float(compute_robust_hausdorff(surface_original, 95)),
        'hd95_clean': float(compute_robust_hausdorff(surface_cleaned, 95)),
        'hd100_orig': float(compute_robust_hausdorff(surface_original, 100)),
        'hd100_clean': float(compute_robust_hausdorff(surface_cleaned, 100)),
    }


def compute_validation_metrics(ct, ground_truth, outputs):
    ground_truth_arr = sitk.GetArrayFromImage(ground_truth)
    spacing = tuple(reversed(ct.GetSpacing()))
    row = {}
    for side, label_value in (('right', 3), ('left', 2)):
        ground_truth = ground_truth_arr == label_value
        mask_original = sitk.GetArrayFromImage(outputs[side]['raw']) > 0
        mask_cleaned = sitk.GetArrayFromImage(outputs[side]['cleaned']) > 0
        acetabular_window = sitk.GetArrayFromImage(outputs[side]['window']) > 0
        side_results = compute_side_metrics(
            ground_truth,
            mask_original,
            mask_cleaned,
            acetabular_window,
            spacing,
        )
        row.update({
            f'{side}_{metric_name}': value
            for metric_name, value in side_results.items()
        })
    return row


In [ ]:
validation_metrics_by_dataset = {}

for dataset_name, root_path in validation_dataset_paths.items():
    print(f'\n=== validation dataset: {dataset_name} ===')
    rows = []
    for subfolder in sorted(os.listdir(root_path)):
        subfolder_path = os.path.join(root_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue
        print(f'\nprocessing: {dataset_name}/{subfolder}')
        row = {'case': subfolder}
        try:
            (
                ct, ground_truth, outputs,
                right_hip_mask_cleaned,
                right_hip_mask_cleaned_fallback,
                right_hip_pred_arr_rev_bi,
                left_hip_mask_cleaned,
                left_hip_mask_cleaned_fallback,
                left_hip_pred_arr_rev_bi,
                right_added_voxels, left_added_voxels,
                right_inf, right_sup, left_inf, left_sup,
            ) = run_cleaning_workflow(subfolder_path)
            df_voxel_count = compute_voxel_count_diff(
                right_hip_mask_cleaned,
                right_hip_mask_cleaned_fallback,
                right_hip_pred_arr_rev_bi,
                left_hip_mask_cleaned,
                left_hip_mask_cleaned_fallback,
                left_hip_pred_arr_rev_bi,
                right_added_voxels,
                left_added_voxels,
                thresh_fallback_replace=60,
                thresh_orig_lower=-10,
                thresh_orig_upper=200,
            )
            df_voxel_count.to_csv(
                os.path.join(subfolder_path, 'voxel_count_diffs.csv'),
                index=False,
            )
            row.update(
                compute_validation_metrics(
                    ct, ground_truth, outputs
                )
            )
            row['status'] = 'ok'
        except Exception as error:
            row['status'] = 'error'
            row['error'] = f'{type(error).__name__}: {error}'
            print(row['error'])
        rows.append(row)

    dataframe = pd.DataFrame(rows)
    validation_metrics_by_dataset[dataset_name] = dataframe
    dataframe.to_csv(
        os.path.join(root_path, 'validation_metrics.csv'), index=False
    )
    print(dataframe)
